# D01 · 口径与底座

这一本不下结论，只把后面五本要用的"尺子"先钉死：
**问什么、用哪些数、哪些数不能用、哪段时间不能拿来比。**

| 本 | 要回答的问题 |
| --- | --- |
| D02 | 客人从哪来 |
| D03 | 什么时候来 |
| D04 | 房型和客源、酒店类型是什么关系 |
| D05 | 取消和什么有关 |
| D06 | 哪些取消是结构性的、哪些真能干预 |

先做这一步的理由很实际：这份数据有三个硬伤（窗口不完整、没有酒店地区字段、没有房型字典），
不先把它们说出来，后面每一张图都可能被读成错误的业务结论。

## 0 环境准备

固定模板：路径一律从 `config` 来，所以 notebook 在哪台机器、哪个目录打开都能找到库。

In [ ]:
%autoreload 2
%matplotlib inline

import sqlite3
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from hotel_insight import config, viz
from hotel_insight.viz import FAINT, INK, MAIN, MAIN_LIGHT, MAIN_MID, MUTED  # 配色常量
from hotel_insight.viz.theme import FONTS

viz.apply_theme()                                  # 主题不随内核保存，每开一次 notebook 都要跑
pd.set_option("display.width", 200)
pd.set_option("display.unicode.east_asian_width", True)

con = sqlite3.connect(config.DB_PATH)              # DB_PATH 由 config 用标记文件定位，不写死路径
NB = "D01"
SOURCE = "数据源：hotel.db / dwd_bookings（已清洗的入住记录）"

In [ ]:
# ---- 自检：版本、字体、库文件对不对 ----
print(f"Python      {sys.version.split()[0]}")
print(f"pandas      {pd.__version__}")
print(f"matplotlib  {matplotlib.__version__}")
print(f"中文字体    {'、'.join(FONTS)}")
print(f"数据库      {config.DB_PATH}")
print(f"参数版本    {config.load_settings()['version']}")

assert config.DB_PATH.exists(), f"找不到数据库 {config.DB_PATH}"
assert any("CJK" in f for f in FONTS), "中文字体没生效，检查内核是不是 signal 环境"

## 1 从 119,390 条到 85,585 条

原始 `bookings` 表 119,390 行，四步清洗后进 `dwd_bookings` 的只有 85,585 行。
顺序不能换，原因写在 `dwd.py` 里：先剔无人入住和零晚数，再剔房价异常，
最后才去完全重复——反过来的话，异常行会先污染"重复"的判定。

这一步的产物已经落在 `ads_clean_steps`，这里只读回来核对，不重算。

In [ ]:
d = pd.read_sql("SELECT * FROM dwd_bookings", con)
steps = pd.read_sql("SELECT * FROM ads_clean_steps", con)
raw_n = config.load_settings()["source"]["expected_rows"]

print(f"清洗后 {len(d):,} 行 × {len(d.columns)} 列；取消率 {d['is_canceled'].mean():.2%}")
print(f"日期范围 {d['arrival_date'].min()} — {d['arrival_date'].max()}")
print(f"原始 {raw_n:,} → 保留 {len(d):,}（{len(d) / raw_n:.1%}）")
steps

In [ ]:
# ── 出图 D01_01：四步清洗的剩余行数 ──
# 为什么用横向条形而不是漏斗图：漏斗图的宽度没有刻度，读者只能看出"在缩"，
# 看不出每步缩了多少；条形带数值，可以直接对账。
# 规范落在哪几行：条末写千分位 / 剔除数写在轴标签第二行 / 最后一步用主色强调
# ---- 1 指标：标题里的数字全部由代码算 ----
dropped = raw_n - len(d)
dup_drop = int(steps.loc[steps["步骤"] == "完全重复", "剔除行数"].iloc[0])
raw_rate = float(pd.read_sql("SELECT AVG(is_canceled) AS r FROM bookings", con)["r"].iloc[0])
labels = ["原始入库"] + [f"{s}\n−{n:,}" for s, n in zip(steps["步骤"], steps["剔除行数"])]
values = [raw_n] + steps["剩余行数"].tolist()
title = f"四步剔掉 {dropped:,} 条（{dropped / raw_n:.1%}），其中 {dup_drop / dropped:.0%} 是重复单"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = (f"完全重复一步剔 {dup_drop:,} 条，占全部剔除量的 {dup_drop / dropped:.0%}；"
            f"含重复的取消率 {raw_rate:.0%}，去重后 {d['is_canceled'].mean():.0%}")
# ---- 2 画图：五根条，从原始到最后一步 ----
fig, ax = plt.subplots(figsize=(9.4, 4.2))
viz.hbar(ax, labels, values, value_fmt="{:,.0f}",
         colors=[viz.GRAY_DARK] + [viz.BLUES[3]] * (len(values) - 2) + [viz.MAIN])
ax.xaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")     # 千分位：120000 与 120,000 差一个人眼停顿
# ---- 3 收尾 ----
viz.finish(fig, title, subtitle, SOURCE, "D01_01_四步清洗剩余行数.png")
# ---- 4 读回断言：每根条的长度必须等于表里的剩余行数 ----
widths = [b.get_width() for b in ax.patches]
assert widths == values, f"条长 {widths} 与表 {values} 不一致"
assert f"{dropped:,}" in fig.texts[0].get_text(), "标题里的剔除数不是代码算的"
plt.show()

结论：**重复记录是这份数据最大的质量问题**——26.6% 的行是完全重复，而且重复行里取消单占六成，
所以不做去重会把取消率系统性高估（含重复 37% vs 去重 28%，差 9 个百分点）。

这是"能不能直接用"和"怎么报数"的分界：`bookings` 保留原样、`dwd_bookings` 已去重，
后面所有分析只认 `dwd_bookings`，报数时不再出现"含重复"的口径。

## 2 时间窗口不完整，这是最容易被读错的一处

27 个月里，**只有 2016 年是完整的自然年**：2015 从 7 月开始，2017 到 8 月结束。
先看这张覆盖图，再决定能不能做同比。

In [ ]:
# ── 出图 D01_02：年 × 月记录数热力 ──
# 为什么用热力矩阵而不是折线：折线会把"没有数据"画成 0，看起来像"那几个月没生意"，
# 而真相是"那几个月不在数据里"。格子里打灰、写字，才能区分"零"和"无"。
# 规范落在哪几行：单色蓝渐变 / 空格打灰写"—" / 千分位
# ---- 1 指标 ----
cover = d.pivot_table(index="arrival_year", columns="arrival_month_no",
                      values="is_canceled", aggfunc="size").reindex(columns=range(1, 13))
cover.columns = [f"{m} 月" for m in cover.columns]
missing = int(cover.isna().sum().sum())
title = f"27 个月里只有 2016 是完整年，另有 {missing} 个格子没有数据"
assert len(title) <= 40
subtitle = "格子为当月记录数；空格（—）表示该月不在数据范围内，不是当月零订单"
# ---- 2 画图 ----
fig, ax = plt.subplots(figsize=(9.6, 3.6))
im = viz.heat(ax, cover, fmt="{:,.0f}", fig=fig, label_size=8)
# ---- 3 收尾 ----
viz.finish(fig, title, subtitle, SOURCE, "D01_02_年月覆盖热力.png")
# ---- 4 读回断言：被遮住的格子数必须等于缺失月份数 ----
arr = im.get_array()
assert int(np.ma.count_masked(arr)) == missing == 10, f"遮住的格子 {np.ma.count_masked(arr)}，应为 10"
plt.show()

In [ ]:
# ---- 对账：三段时间必须拼回全部记录，且只有 2016 是 12 个月 ----
by_year = d.groupby("arrival_year").size()
months_of_year = d.groupby("arrival_year")["arrival_month_no"].nunique()

assert by_year.sum() == len(d)
assert months_of_year.loc[2016] == 12, "2016 应是唯一完整的 12 个月"
assert months_of_year.loc[2015] == 6 and months_of_year.loc[2017] == 8

print(f"2015 下半年  {by_year[2015]:>6,} 条（{months_of_year[2015]} 个月）")
print(f"2016 全年    {by_year[2016]:>6,} 条（{months_of_year[2016]} 个月）← 唯一完整年")
print(f"2017 前八月  {by_year[2017]:>6,} 条（{months_of_year[2017]} 个月）")
print(f"合计         {by_year.sum():>6,} 条")
print("✅ 对账通过：三段相加等于清洗后总行数，且只有 2016 可做整年口径")

所以后面所有的"同比"都只能用 **2016 与 2017 的 1—8 月**（`timing.comparable_slice`）。
拿 2016 全年比 2017 前八个月，少掉的 4 个月会被读成"业务下滑"，这是这份数据最容易踩的坑。

季节指数也只能用 2016 算——用不完整的年份算，1 月会凭空变成淡季，
所以 `timing.season_index()` 直接用 `raise` 拒绝非完整年份。

## 3 两条业务线：城市店与度假店

两家酒店的客源结构、价格、取消率都不同，后面每一本都要分线看，
否则"平均"会把两条不同的曲线压成一条没有意义的线。

In [ ]:
line = d.groupby("hotel_type").agg(
    记录数=("is_canceled", "size"), 取消率=("is_canceled", "mean"),
    均价=("adr", "mean"), 平均停留=("total_nights", "mean"),
    提前期中位=("lead_time", "median"))
line["占比"] = line["记录数"] / len(d)
line

In [ ]:
# ── 出图 D01_03：两条业务线的规模与取消率 ──
# 为什么用两个子图而不是双轴：记录数与取消率量纲不同，双轴会让柱子高度与标记位置
# 产生假的对应关系；并排放两把尺子，读者不会看串。
# 规范落在哪几行：同一对象同一颜色（HOTEL_COLOR）/ 取消率带 95% Wilson 区间 / 横轴百分比
# ---- 1 指标：标题里的数字全部由代码算 ----
from hotel_insight import rate

lines = line.index.tolist()
n_vals = line["记录数"].to_numpy()
r_vals = line["取消率"].to_numpy()
# 区间按"单数/行数"重算，不对比率做正态近似：样本越小，两种算法差得越多
ks = d.groupby("hotel_type")["is_canceled"].sum().reindex(lines).to_numpy()
lo, hi = rate.wilson(ks, n_vals)          # 返回两个数组，一次算完两组
lo, hi = np.atleast_1d(lo), np.atleast_1d(hi)

city, resort = line.loc["城市酒店"], line.loc["度假酒店"]
title = f"城市店取消率 {city['取消率']:.1%}，比度假店高 {(city['取消率'] - resort['取消率']) * 100:.1f} 个百分点"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = (f"城市店 {int(city['记录数']):,} 条、度假店 {int(resort['记录数']):,} 条；"
            f"误差线为 95% Wilson 区间，两组区间不重叠")

# ---- 2 画图 ----
fig, axes = plt.subplots(1, 2, figsize=(10.2, 3.4))
colors = [viz.HOTEL_COLOR[x] for x in lines]
viz.hbar(axes[0], lines, n_vals, value_fmt="{:,.0f}", colors=colors)
axes[0].xaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
axes[0].set_title("记录数（条）", fontsize=10, color=MUTED, loc="left")
viz.hbar(axes[1], lines, r_vals, value_fmt="{:.1%}", colors=colors,
         errors=(lo, hi))
axes[1].xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
axes[1].set_xlim(0, r_vals.max() * 1.28)     # 留白给条末标签与误差线，否则会被右边界裁掉
axes[1].tick_params(labelleft=False)         # 右图与左图同一个 y 轴，标签不重复写两遍
axes[1].set_title("取消率（误差线为 95% 区间）", fontsize=10, color=MUTED, loc="left")

# ---- 3 收尾 ----
viz.finish(fig, title, subtitle, SOURCE, "D01_03_两条业务线.png")
# ---- 4 读回断言：右图每根条的长度必须等于分组取消率，标题数字必须来自代码 ----
bars = axes[1].patches
assert np.allclose([b.get_width() for b in bars], r_vals), "条长与取消率不一致"
assert lo[0] > hi[1], "两组区间有重叠，副标题里'不重叠'这句话就不成立"
assert f"{city['取消率']:.1%}" in fig.texts[0].get_text(), "标题数字不是代码算的"
plt.show()

城市店占 61% 的记录、取消率却高 6.7 个百分点。这个差距本身还不是结论——
它可能来自客源结构（城市店更多 OTA），也可能来自房价（城市店更贵）。
D02 与 D05 会分别拆开看，这里只记住一条：**后面所有指标都要分线报**。

## 4 口径卡

后面五本沿用同一套口径，改口径就要回来改这里。

| 项目 | 本项目的口径 | 依据 |
| --- | --- | --- |
| 分析对象 | `dwd_bookings`，85,585 条 | 四步清洗后，见 `ads_clean_steps` |
| 时间窗 | 2015-07 至 2017-08，共 27 个月 | 数据本身的范围 |
| 唯一完整年 | 2016（12 个月） | DQ-01 |
| 同比窗口 | 2016 与 2017 的 1—8 月 | 两年都有这 8 个月 |
| 取消率 | 取消单数 ÷ 记录数，附 95% Wilson 区间 | `rate.rate_table` |
| 分组样本下限 | 30 条；不足只画点不给结论 | `settings.analysis.min_group_n` |
| 矩阵格子下限 | 20 条；不足打灰 | `settings.analysis.min_cell_n` |
| 收入口径 | `adr × total_nights`，是**名义**收入 | 没有每日房态，算不了真实损失 |
| 房型 | 只有 A—P 字母代码，**没有名称对照表** | DQ-16 |
| 地区 | 只有客源国，**没有酒店所在地区字段** | 见下节 |

**两条必须记住的"答不了"**：

1. 这份数据没有酒店所在城市/地区字段。所以"地区与房型的关系"只能答成
   **客源国 × 房型**，不能答成"酒店所在地 × 房型"。任何写成"某地区客源多"的结论都是超范围。
2. 没有房型字典，A—P 对应什么房型未知。所以房型结论只能按代码比较（A 比 H 便宜），
   不能翻译成"标准间比套房便宜"。

## 5 发现清单（本册）

| # | 发现 | 影响 | 处理 |
| --- | --- | --- | --- |
| 1 | 完全重复占 26.6%，且重复行取消率更高 | 不去重会高估取消率约 9pp | 分析只用 `dwd_bookings` |
| 2 | 2015 只有下半年、2017 只有前八个月 | 不能做整年同比；季节指数只能用 2016 | 同比固定取 1—8 月 |
| 3 | 没有酒店所在地区字段 | "地区"问题只能答到客源国 | 在 D02/D04 里改写问题口径 |
| 4 | 城市店取消率高 6.7pp | 合看会掩盖结构差异 | 所有指标分线报 |

还有一条留给后面的册子：数据的 `required_car_parking_spaces ≥ 1` 时取消率**恰好为 0**，
`deposit_type = 不可退款` 时取消率 **94.7%**——这两条都不像真实业务，D05 会专门查。

In [ ]:
con.close()
print("D01 完成：口径已钉死，后面五本不再重复解释")